# SpaceX Falcon 9 — Interactive Dashboard

IBM Data Science Capstone | Plotly Dash

## Libraries

I used pandas for the launch data and Dash with Plotly for the interactive charts.

In [7]:
%pip install -q pandas plotly dash

## Dataset overview

This dashboard uses the historical launch dataset prepared earlier in the project. I checked the class counts and launch sites before creating the charts. The 2025 Wikipedia scraping dataset is separate and is not used here.

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_PATH = Path("spacex_launches_processed.csv")
if not DATA_PATH.exists():
    try:
        from google.colab import files
        print("Upload spacex_launches_processed.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put spacex_launches_processed.csv in the working folder.")

df = pd.read_csv(DATA_PATH)
required = {"FlightNumber", "LaunchSite", "PayloadMass", "Orbit", "BoosterVersion", "Class"}
assert required.issubset(df.columns), f"Missing columns: {required-set(df.columns)}"
assert set(df["Class"].dropna().unique()).issubset({0,1})
print("Shape:", df.shape)
print("Class counts:\n", df["Class"].value_counts().sort_index())
print("Launch sites:\n", df["LaunchSite"].value_counts())
display(df.head())

Upload spacex_launches_processed.csv


Saving spacex_launches_processed.csv to spacex_launches_processed.csv
Shape: (90, 18)
Class counts:
 Class
0    11
1    79
Name: count, dtype: int64
Launch sites:
 LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,1
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,1
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,1
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,1


## Dashboard development

I built a launch-site dropdown, a pie chart of the two landing classes, and a payload-range slider linked to a scatter plot. The callbacks update the figures whenever a filter changes. This cell saves the same dashboard as a standalone Python script.

In [3]:
from pathlib import Path

DASH_SOURCE = 'import pandas as pd\nimport plotly.express as px\nfrom pathlib import Path\nfrom dash import Dash, dcc, html, Input, Output\n\nDATA_FILE = Path(__file__).resolve().parent / "spacex_launches_processed.csv"\ndf = pd.read_csv(DATA_FILE)\nrequired = {"FlightNumber", "LaunchSite", "PayloadMass", "Orbit", "BoosterVersion", "Class"}\nmissing = required - set(df.columns)\nif missing:\n    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")\n\napp = Dash(__name__)\napp.title = "SpaceX Falcon 9 Analytics"\napp.layout = html.Div([\n    html.H1("SpaceX Falcon 9 Launch Analytics Dashboard", style={"textAlign": "center"}),\n    html.P("Historical launch classifications (IBM course labels)", style={"textAlign": "center"}),\n    html.P(f"Records: {len(df)} | Class 1: {(df[\'Class\']==1).sum()} | Class 0: {(df[\'Class\']==0).sum()}",\n           style={"textAlign": "center"}),\n    html.Label("Select Launch Site"),\n    dcc.Dropdown(id="site-dropdown", options=[{"label":"All Launch Sites","value":"ALL"}]\n                 + [{"label":s,"value":s} for s in sorted(df["LaunchSite"].dropna().unique())],\n                 value="ALL", clearable=False),\n    dcc.Graph(id="classification-pie-chart"),\n    html.Label("Select Payload Range (kg)"),\n    dcc.RangeSlider(id="payload-slider", min=0, max=16000, step=1000,\n                    value=[0,16000], marks={i:str(i) for i in range(0,16001,4000)}),\n    dcc.Graph(id="payload-scatter-chart"),\n    html.P("Note: Class 1 follows the course convention; it does not independently verify a successful landing.",\n           style={"fontSize":"12px", "color":"#555"})\n], style={"maxWidth":"1100px", "margin":"auto", "padding":"20px", "fontFamily":"Arial, sans-serif"})\n\n@app.callback(Output("classification-pie-chart","figure"), Input("site-dropdown","value"))\ndef update_classification_pie(selected_site):\n    subset = df if selected_site == "ALL" else df[df["LaunchSite"] == selected_site]\n    counts = subset["Class"].value_counts().reindex([0,1], fill_value=0).reset_index()\n    counts.columns = ["Class","Count"]\n    counts["Classification"] = counts["Class"].map({0:"Class 0",1:"Class 1"})\n    fig = px.pie(counts, values="Count", names="Classification",\n                 title=f"Class Distribution — {(\'All Launch Sites\' if selected_site==\'ALL\' else selected_site)}",\n                 color="Classification", color_discrete_map={"Class 0":"red","Class 1":"green"})\n    fig.update_traces(textinfo="percent+label", hovertemplate="%{label}: %{value} launches (%{percent})<extra></extra>")\n    return fig\n\n@app.callback(Output("payload-scatter-chart","figure"),\n              Input("site-dropdown","value"), Input("payload-slider","value"))\ndef update_payload_scatter(selected_site, payload_range):\n    subset = df.copy() if selected_site == "ALL" else df[df["LaunchSite"] == selected_site].copy()\n    subset = subset[subset["PayloadMass"].between(payload_range[0],payload_range[1])].copy()\n    subset["Classification"] = subset["Class"].map({0:"Class 0",1:"Class 1"})\n    fig = px.scatter(subset, x="PayloadMass", y="FlightNumber", color="Classification",\n                     color_discrete_map={"Class 0":"red","Class 1":"green"},\n                     hover_data=["LaunchSite","Orbit","BoosterVersion"],\n                     title="Flight Number vs Payload Mass by Classification",\n                     labels={"PayloadMass":"Payload Mass (kg)","FlightNumber":"Flight Number"})\n    fig.update_layout(legend_title="Classification")\n    return fig\n\nif __name__ == "__main__":\n    app.run(debug=False, port=8050)\n'
Path("spacex_dashboard.py").write_text(DASH_SOURCE, encoding="utf-8")
print("Saved spacex_dashboard.py", Path("spacex_dashboard.py").stat().st_size, "bytes")

Saved spacex_dashboard.py 3554 bytes


## Check the exported script

I checked the Python syntax and the presence of both Dash callbacks before running the application locally.

In [4]:
import ast
source = Path("spacex_dashboard.py").read_text(encoding="utf-8")
ast.parse(source)
assert source.count("@app.callback") == 2
assert "html.Div" in source and "dcc.Graph" in source
assert 'if __name__ == "__main__":' in source
print("PASS: valid Python syntax, Dash layout and both callbacks are present")

PASS: valid Python syntax, Dash layout and both callbacks are present


## Download the application

The Python file can be run in VS Code alongside `spacex_launches_processed.csv`. This download cell is optional.

In [5]:
# Optional: download the standalone application from Colab
# from google.colab import files
# files.download("spacex_dashboard.py")
print("The dashboard script is saved as spacex_dashboard.py")

The dashboard script is saved as spacex_dashboard.py


## Running the dashboard

I ran the application locally in VS Code because the Colab proxy did not display it reliably. With the script and CSV in the same folder, I installed `pandas`, `plotly`, and `dash`, then ran `python spacex_dashboard.py` and opened `http://127.0.0.1:8050/`.

The full dataset has 79 Class 1 and 11 Class 0 records (87.8% and 12.2%). These labels follow the IBM course convention; Class 1 should not be interpreted as independently confirmed successful landings for every record. The dashboard makes it easier to compare these labels across launch sites and payload ranges.

## Chart preview

I also plotted the two figures here to check their appearance before testing the interactive filters in the local Dash application.

In [6]:
import plotly.express as px

counts = df["Class"].value_counts().reindex([0, 1], fill_value=0).reset_index()
counts.columns = ["Class", "Count"]
counts["Label"] = counts["Class"].map({0: "Class 0", 1: "Class 1"})
fig = px.pie(counts, values="Count", names="Label", title="Landing Classification — All Sites",
             color="Label", color_discrete_map={"Class 0": "red", "Class 1": "green"})
fig.show()

plot_data = df.copy()
plot_data["Label"] = plot_data["Class"].map({0: "Class 0", 1: "Class 1"})
fig = px.scatter(plot_data, x="PayloadMass", y="FlightNumber", color="Label",
                 hover_data=["LaunchSite", "Orbit"],
                 color_discrete_map={"Class 0": "red", "Class 1": "green"},
                 title="Flight Number vs Payload Mass")
fig.show()